### Background

This Jupyter Notebook demonstrates how we generated additional data to expand our training set. The main goal of data generation was to address class imbalance and enhance the diversity of our dataset, ultimately improving the performance of our model. We used the OpenAI API as the model provider for data generation.

Before start, you need to install the following dependencies:

```
python-dotenv
openai
pandas
numpy
tqdm
```

Commanda if you use conda environment:

```
conda activate <env-name>
conda install pandas numpy -y
pip install python-dotenv
pip install openai
conda install conda-forge::tqdm
```

Create `.env` file with `OPENAI_API_KEY`

### Imports

In [26]:
import os

import numpy as np
import pandas as pd

from tqdm import tqdm
from dotenv import load_dotenv

from openai import OpenAI

### Init OpenAI client

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

### Constants

In [4]:
TRAIN_PATH = "data/"
TRAIN_NAME = "train.parquet"

### Read Data

In [5]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.shape

(3822, 6)

In [6]:
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [7]:
df = df.loc[~df["techniques"].isna()]
df.shape

(2589, 6)

### Helpers

In [8]:
def extract_phrases(content, trigger_words):
    """
    Extracts phrases from the content based on trigger word indices.

    :param content: The input text.
    :param trigger_words: List of index ranges [[start1, end1], [start2, end2], ...].
    :return: List of extracted phrases.
    """
    return [content[start:end] for start, end in trigger_words]

In [78]:
def check_spans_in_text(text: str, spans: list[str]) -> tuple[dict[str, bool], float]:
    """
    Check if each span in the list is a substring of the given text.
    """
    results = {span: span in text for span in spans}
    true_count = sum(results.values())  # Count how many spans are found
    success_rate = true_count / len(spans) if spans else 0.0  # Avoid division by zero
    
    return results, success_rate

# Example usage:
text_sample = "Artificial intelligence is transforming the world."
spans_list = ["Artificial intelligence", "Machine learning", "transforming"]

result_dict, rate = check_spans_in_text(text_sample, spans_list)

round(rate, 2), result_dict

(0.67,
 {'Artificial intelligence': True,
  'Machine learning': False,
  'transforming': True})

In [100]:
def find_spans_in_text(text: str, spans: list[str]) -> list[list[int]]:
    """
    Find the start and end indices of spans within the text.
    """
    found_spans = []
    
    for span in spans:
        start_idx = text.find(span)  # Find first occurrence
        if start_idx != -1:
            end_idx = start_idx + len(span)  # Compute end index
            found_spans.append([start_idx, end_idx])
    
    return found_spans

### Prompting

In [134]:
def generate_prompt(text: str, triggers: list[str]):
    prompt = f"""You are telegram editor. You need to create a unique samples based on the given text.

Rephrase or generate a similar text to the given text [TEXT].
IMPORTANT: Leave the phrases defined in list [UNCHANGED] exactly the same (do not highlight them please).

Be creative so the text is not very similar to the given one.
Preserve the meaning and the style of the text. Also, make sure you are using the same language and style as the original text.

[TEXT]
{text}

[UNCHANGED]
{triggers}"""
    return prompt

In [43]:
def generate_sample(prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        store=True,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return completion.choices[0].message.content

In [44]:
def generate_sample(prompt: str):
    completion = client.chat.completions.create(
        model="gpt-4o",
        store=True,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return completion.choices[0].message.content

In [103]:
example = df.sample(1)

techniques = example["techniques"].iloc[0]
text = example["content"].iloc[0]
triggers = extract_phrases(text, example["trigger_words"].iloc[0])
print(example["trigger_words"].iloc[0])
prompt = generate_prompt(text, triggers)

print(techniques)
print("-" * 50)
print(text)
print("-" * 50)
print(triggers)
print("-" * 50)
print(prompt)

[array([233, 284]) array([288, 370])]
['cliche' 'loaded_language']
--------------------------------------------------
В картине мира каждого из нас именно он - центр Вселенной. 
В картине мира большинства моих соотечественников Украина - центр мира. 
Все из-за нас и для нас. 
Но объективно мир устроен несколько иначе. И он (мир) меняется. 
Говорят, не повезло тем, кому выпало жить во времена перемен.  
О том кому «повезло» больше, кому меньше и что будет дальше - в следующем выпуске.
https://youtu.be/ijOpn6S5kTE
--------------------------------------------------
['не повезло тем, кому выпало жить во времена перемен', 'О том кому «повезло» больше, кому меньше и что будет дальше - в следующем выпуске.']
--------------------------------------------------
You are telegram editor. You need to create a unique samples based on the given text.

Rephrase or generate a similar text to the given text [TEXT].
Leave the phrases defined in list [UNCHANGED] exactly the same (do not highlight them ple

In [104]:
%%time

generated_sample = generate_sample(prompt)
print(generated_sample)
print()

Для каждого человека его личная картина мира всегда ставит его в центр своей Вселенной. Для многих из моих сограждан Украина представляется центром глобального пространства. Всё происходит из-за нас и ради нас. Однако объективно мир функционирует иначе и продолжает изменяться. Как говорится, не повезло тем, кому выпало жить во времена перемен. О том кому «повезло» больше, кому меньше и что будет дальше - в следующем выпуске.

CPU times: user 15.4 ms, sys: 4.24 ms, total: 19.6 ms
Wall time: 2.71 s


In [106]:
res_dict, res_rate = check_spans_in_text(generated_sample, triggers)

print(round(res_rate, 2))
print(res_dict)

1.0
{'не повезло тем, кому выпало жить во времена перемен': True, 'О том кому «повезло» больше, кому меньше и что будет дальше - в следующем выпуске.': True}


In [107]:
find_spans_in_text(generated_sample, triggers)

[[293, 344], [346, 428]]

### Generation

In [110]:
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"
5,46493f44-f00a-4ffb-9cda-252ccf5fa4c6,"Апартаменти\n триповерхова келія Паші Лєбєдя, ...",uk,True,[loaded_language],"[[94, 108], [208, 227]]"


In [135]:
data = df.sample(20)

result = []
for index, row in tqdm(data.iterrows(), total=len(data)):
    try:
        triggers = extract_phrases(row["content"], row["trigger_words"])
        prompt = generate_prompt(row["content"], triggers)
        
        generated_content = generate_sample(prompt)
        # generated_content = ""
        generation_checker, generation_rate = check_spans_in_text(generated_content, triggers)

        generated_trigger_words = find_spans_in_text(generated_content, triggers)
    
        result.append({
            "id": row["id"], # id of original sample
            "prompt": prompt,
            "generated_content": generated_content,
            "trigger_words_string": triggers,
            "generated_trigger_words": generated_trigger_words,
            "generation_rate": round(generation_rate, 2),
            "generation_checker": generation_checker,
        })
    except e:
        print(e)

100%|████████████████████████████████████████████████████████████████████████████| 20/20 [01:34<00:00,  4.74s/it]


In [137]:
#  id  - id of original sample
#  prompt - used prompt
#  generated_content - generated content
#  trigger_words_string - list of spans that need to be without changes
#  generated_trigger_words - span location in generated text
#  generation_rate - quality of generated text: if you can match only 2 of 3 spans, rate is 2/3 ~ 0.67
#  generation_checker - dict of true/false of detected spans for debuging

generated_df = pd.DataFrame(result)
generated_df.head(10)

,id,prompt,generated_content,trigger_words_string,generated_trigger_words,generation_rate,generation_checker
0,2f7f7d03-302b-49b9-bc2b-d9e4d87f588f,You are telegram editor. You need to create a ...,😳 \nРоздавати повістки тепер будуть на роботі...,"[Роздавати повістки тепер будуть на роботі, З ...","[[4, 45], [517, 562]]",1.00,{'Роздавати повістки тепер будуть на роботі': ...
1,372e4fa3-c38e-42e9-b46f-697c721a679f,You are telegram editor. You need to create a ...,▪️ \nМинулої ночі ворог здійснив атаку на Нік...,"[Доки ми за традицією наводимо в оселях лад, н...","[[387, 531]]",1.00,"{'Доки ми за традицією наводимо в оселях лад, ..."
2,57d9f8d7-dbc7-4f7e-b793-0250bed8ed5e,You are telegram editor. You need to create a ...,👍\nСотрудники следственного управления совмест...,"[Ребята были очень рады гостям., от чистого се...","[[486, 516], [647, 693]]",1.00,"{'Ребята были очень рады гостям.': True, 'от ч..."
3,53464434-2c08-43d1-8607-150fb9fcf206,You are telegram editor. You need to create a ...,☝️ \nНезламним є навіть зруйнований ворогом м...,[Незламним є навіть зруйнований ворогом міст ч...,"[[5, 60]]",1.00,{'Незламним є навіть зруйнований ворогом міст ...
4,e41f8147-083f-4440-a62a-9700e28635ac,You are telegram editor. You need to create a ...,"⚡️ \nНачиная с 20 июля, все суда, направляющи...","[украинские порты, будут рассматриваться как п...","[[300, 316]]",0.33,"{'украинские порты, будут рассматриваться как ..."
5,dd55f01b-47aa-4238-b994-fae657720465,You are telegram editor. You need to create a ...,🗣\nУхвалене рішення щодо оподаткування фінансо...,"[Зрозуміло, що заплатять звичайні люди…]","[[140, 178]]",1.00,"{'Зрозуміло, що заплатять звичайні люди…': True}"
6,a944b7c6-112b-4794-997c-289457199909,You are telegram editor. You need to create a ...,⚡️ Вперше на екрані! Запрошую вас на четвертий...,"[Коли знищать Кримський міст, Хусити зрадили р...","[[221, 248], [310, 331]]",1.00,"{'Коли знищать Кримський міст': True, 'Хусити ..."
7,9e099e53-7112-44f4-92f5-64dba7416e80,You are telegram editor. You need to create a ...,"Тільки уявіть, можна отримати 2 статуси одноча...",[уявіть це виходить 2 статуси одразу -учасник ...,[],0.00,{'уявіть це виходить 2 статуси одразу -учасник...
8,03d62912-1f02-4c68-aa7c-c7c5f9d4689a,You are telegram editor. You need to create a ...,Министерство обороны России демонстрирует виде...,"[товарищи, хохлы много, чего тоже заявляют, у ...","[[114, 122], [209, 292], [294, 355]]",1.00,"{'товарищи': True, 'хохлы много, чего тоже зая..."
9,38846ac2-3262-462e-9727-891e09505aeb,You are telegram editor. You need to create a ...,"Самое время открыть рубрику ""нестыдно"". Перелё...","[Пора вводить рубрику ""нестыдно"", можно в фурш...","[[82, 155]]",0.50,"{'Пора вводить рубрику ""нестыдно""': False, 'мо..."


In [138]:
generated_df.to_csv("data/generated_sample.csv", index=False)

In [139]:
generated_df["generation_rate"].mean()

np.float64(0.785)

In [123]:
# de-bugging
# for _, row in generated_df.iterrows():
#     print(row["generated_content"])